<a href="https://colab.research.google.com/github/Karna14314/Hugginface_models/blob/main/MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Professional Mini Image Classifier: MobileNetV2
This notebook performs transfer learning on a custom dataset and deploys the model to Hugging Face Spaces.

In [ ]:
!pip install -q gradio huggingface_hub tensorflow

## 1. Setup & Authentication
Please ensure you have a Hugging Face token with 'Write' access stored in your Colab secrets as `HF_TOKEN`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import userdata
from huggingface_hub import HfApi, login
import getpass

try:
    # Try to get from Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    print("HF_TOKEN secret not found. Please paste your token below:")
    hf_token = getpass.getpass("Hugging Face Token: ")

try:
    login(token=hf_token)
    print("Successfully logged into Hugging Face")
except Exception as e:
    print(f"Login failed: {e}")

## 2. Model Architecture (MobileNetV2)
We will freeze the base layers and add a custom classifier head.

In [ ]:
IMG_SIZE = (224, 224)

def build_model(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = models.Sequential([
        # Add augmentation here to prevent overfitting
        layers.Input(shape=IMG_SIZE + (3,)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3), # Increased dropout
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model

## 4. Data Preparation
We will use a small flower dataset as a placeholder for your custom data.

In [ ]:
import pathlib
import tensorflow as tf
from tensorflow.keras import layers

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos_dir', origin=dataset_url, untar=True)
# The dataset extracts into a 'flower_photos' directory inside the destination
data_dir = pathlib.Path(data_dir) / 'flower_photos'

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=32,
  label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=32,
  label_mode='categorical'
)

class_names = train_ds.class_names
print(f"Classes found: {class_names}")

## 5. Training
Training the custom classifier head.

In [ ]:
model, base_model = build_model(len(class_names))

print("Starting training...")
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=5
)

model.save('model.h5')
print("Model saved as model.h5")

## 4. Data Preparation
We will use a small flower dataset as a placeholder for your custom data.

In [ ]:
import pathlib
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=32,
  label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=32,
  label_mode='categorical'
)

class_names = train_ds.class_names
print(f"Classes found: {class_names}")

## 5. Training
Training the custom classifier head.

In [ ]:
model, base_model = build_model(len(class_names))

print("Starting training...")
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=5
)

model.save('model.h5')
print("Model saved as model.h5")

## 3. Gradio Interface & Deployment Script
This cell prepares the `app.py` for Hugging Face Spaces.

In [ ]:
import os

def create_hf_space(repo_id="ncncomplete/mini-image-classifier"):
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)

    # Template for app.py logic
    app_code = """
import gradio as gr
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model('model.h5')
labels = ['Class_A', 'Class_B'] # Update this after training

def predict(image):
    image = tf.image.resize(image, (224, 224))
    image = np.expand_dims(image, axis=0) / 255.0
    prediction = model.predict(image)[0]
    return {labels[i]: float(prediction[i]) for i in range(len(labels))}

interface = gr.Interface(fn=predict, inputs='image', outputs='label')
interface.launch()
"""
    with open("app.py", "w") as f:
        f.write(app_code)

    print(f"Space setup initiated for {repo_id}")

## 6. Deployment Execution
Running the deployment to Hugging Face.

In [ ]:
# First, generate the app.py file using the function defined earlier
create_hf_space()

# Now update the app.py with correct labels
labels_str = str(class_names)
with open("app.py", "r") as f:
    content = f.read()
content = content.replace("['Class_A', 'Class_B']", labels_str)
with open("app.py", "w") as f:
    f.write(content)

# Upload everything
api = HfApi()
api.upload_file(
    path_or_fileobj="model.h5",
    path_in_repo="model.h5",
    repo_id="ncncomplete/mini-image-classifier",
    repo_type="space"
)
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id="ncncomplete/mini-image-classifier",
    repo_type="space"
)

print("Deployment complete! Check: https://huggingface.co/spaces/ncncomplete/mini-image-classifier")

In [ ]:
import pathlib

# Create requirements.txt
requirements = """
tensorflow
gradio
numpy
"""
with open("requirements.txt", "w") as f:
    f.write(requirements)

# Upload requirements.txt to the Space
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="ncncomplete/mini-image-classifier",
    repo_type="space"
)

print("requirements.txt uploaded. The Space should now rebuild and start correctly.")

## 6. Deployment Execution
Running the deployment to Hugging Face.

In [ ]:
# Update the app.py with correct labels
labels_str = str(class_names)
with open("app.py", "r") as f:
    content = f.read()
content = content.replace("['Class_A', 'Class_B']", labels_str)
with open("app.py", "w") as f:
    f.write(content)

# Create Space and Upload
create_hf_space()
api = HfApi()
api.upload_file(
    path_or_fileobj="model.h5",
    path_in_repo="model.h5",
    repo_id="ncncomplete/mini-image-classifier",
    repo_type="space"
)
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id="ncncomplete/mini-image-classifier",
    repo_type="space"
)

print("Deployment complete! Check your HF Space.")

## 7. Upload to Hugging Face Model Hub
This pushes the trained weights to a dedicated model repository.

In [ ]:
model_repo_id = "ncncomplete/mini-image-classifier-model"

# Create the model repository
api.create_repo(repo_id=model_repo_id, repo_type="model", exist_ok=True)

# Upload the model file
api.upload_file(
    path_or_fileobj="model.h5",
    path_in_repo="model.h5",
    repo_id=model_repo_id,
    repo_type="model"
)

print(f"Model successfully uploaded to: https://huggingface.co/{model_repo_id}")